In [13]:
import numpy as np
from sympy import symbols, expand, binomial
import re

En esta nueva versión los coeficientes de $g_k$ se calculan de forma directa y no numérica.

In [14]:
ROUND_VALUE = 10

In [15]:
# Función auxiliar para poder tratar el output de la 
# función expand
def transform_key(key):
    key = str(key)

    if key == '1':
        return 0
    
    if key == 'w':
        return 1
    
    match = re.search(r'w\*\*([0-9]+)', key)
    return int(match.group(1))
    
    

In [16]:
w = symbols('w')
def calculate_coefficients_of_gk(k):
    '''
    Devuelve coeficientes de la serie de cosenos de g_k
    '''
    g_k = 0

    for l in range(k + 1):
        g_k += binomial(2 * k + 1, k - l) * (1 - w)**(k - l) * (1 + w)**l

    g_k *= (1 + w)**(k + 1) / (2**(2 * k + 1))

    expanded_gk = expand(g_k).as_coefficients_dict()

  

    coeff_dict = {transform_key(k): v for k, v in 
                  expanded_gk.items()}

    return coeff_dict

In [17]:
def get_conversion_dict(max_exponent):
    '''
        Devuelve un diccionario exponente, lista de exponentes
        con la equivalencia cos(w)^n = sum from -n to n e^ikw
    '''

    # cos(w)^0 = 1 = e^0, cos(w) = 1/2 (e^ikw + e^-ikw)
    conversion_dict = {0: [1], 1: [1/2, 0, 1/2]}

    coef_cos = [1/2, 0, 1/2]

    for k in range(2, max_exponent+1):
        coef_cos = np.convolve(coef_cos, [1/2, 0, 1/2])
        conversion_dict[k] = coef_cos
    
    return conversion_dict

   

def transform_cos_to_exp(coeff_dict, conversion_dict=None):
    '''
    Transforma los coeficientes de la serie de cosenos de a una serie de
    exponenciales
    '''

    # Si llamamos esta función varias veces podemos precomputar 
    # el diccionario de conversión
    if not conversion_dict:
        max_exponent = max(coeff_dict.keys())
        conversion_dict = get_conversion_dict(max_exponent)

    exponential_coefs = [0]*(2*max_exponent+1)

    for l, c in coeff_dict.items():
        j = max_exponent - l
        coefs = [0]*j + [c*v for v in conversion_dict[l]] + [0]*j
        exponential_coefs = [sum(x) for x in zip(exponential_coefs, coefs)]
        


    return exponential_coefs

In [18]:
def get_coefficients_gk(k):
    
    '''
    Calcula los coeficientes de la función g_k 
    cuando se escribe como serie de exp(ikw)
    '''

    cos_coefficients = calculate_coefficients_of_gk(k)
    exp_coefficients = transform_cos_to_exp(cos_coefficients)
    return exp_coefficients


A partir de aquí, no hay ningún cambio con respecto a la versión 1 salvo que las raíces se calculan de forma SIMBÓLICA.

In [19]:
from sympy import Poly
from sympy.abc import x

def get_roots_inside_unit_circle(coefs):
    '''
    Calcula las raíces dentro del circulo unitario
    '''
    p = Poly(coefs,x)
    _roots = p.all_roots() #Cálculo simbólico

    _roots = sorted(_roots, key=abs)
    return _roots[:len(_roots)//2]

Hallamos las raíces de P(z) y nos quedamos con las que están dentro del círculo unidad. Con estas raíces construimos el filtro m. <br>
Los coeficientes de $m(\xi)$ los calculamos a partir de sus raíces. <br>

In [34]:
def get_filter_coefficients(roots):
    '''
    Calcula los coeficientes del filtro a partir de las raices
    h_0 = (-1)^n * r1*r2*...*rn
    ....
    h_(n-1) = -(r1 + r2 + ... + rn)
    h_n = 1

    '''

    coefs = [1]
    for r in roots:
        coefs = np.convolve(coefs, [1, -r])

    f = np.sqrt(2)/sum(coefs)

    #Normalizar 
    return [f*c for c in coefs]

In [21]:
import sympy
def get_Daubechies_filter(k):
    coefs = get_coefficients_gk(k)
    roots = get_roots_inside_unit_circle(coefs)
    filter_coefs = get_filter_coefficients(roots)

    # Los coeficientes han de ser reales. Forzamos la parte imaginaria a 0

    filter_coefs = [sympy.re(c.evalf()) for c in filter_coefs]

    return filter_coefs

In [33]:
# imprimimos valores de los filtros. 
# (k+1 para seguir con la notación de la tabla que aparece en Daubechies Ten Lectures On Wavelets)
for k in range(0, 4):
    print(k+1, get_Daubechies_filter(k))

[1 1]
0.707106781186548
1 [0.707106781186548, 0.707106781186548]
[1 sqrt(3) -3 + 2*sqrt(3) -2 + sqrt(3)]
0.482962913144534
2 [0.482962913144534, 0.836516303737808, 0.224143868042013, -0.129409522551260]
[1
 3 - CRootOf(3*x**4 - 18*x**3 + 38*x**2 - 18*x + 3, 1) - CRootOf(3*x**4 - 18*x**3 + 38*x**2 - 18*x + 3, 0)
 CRootOf(3*x**4 - 18*x**3 + 38*x**2 - 18*x + 3, 0)*CRootOf(3*x**4 - 18*x**3 + 38*x**2 - 18*x + 3, 1) + 3 - 3*CRootOf(3*x**4 - 18*x**3 + 38*x**2 - 18*x + 3, 1) - 3*CRootOf(3*x**4 - 18*x**3 + 38*x**2 - 18*x + 3, 0)
 1 - 3*CRootOf(3*x**4 - 18*x**3 + 38*x**2 - 18*x + 3, 1) + 3*CRootOf(3*x**4 - 18*x**3 + 38*x**2 - 18*x + 3, 0)*CRootOf(3*x**4 - 18*x**3 + 38*x**2 - 18*x + 3, 1) - 3*CRootOf(3*x**4 - 18*x**3 + 38*x**2 - 18*x + 3, 0)
 -CRootOf(3*x**4 - 18*x**3 + 38*x**2 - 18*x + 3, 1) + 3*CRootOf(3*x**4 - 18*x**3 + 38*x**2 - 18*x + 3, 0)*CRootOf(3*x**4 - 18*x**3 + 38*x**2 - 18*x + 3, 1) - CRootOf(3*x**4 - 18*x**3 + 38*x**2 - 18*x + 3, 0)
 CRootOf(3*x**4 - 18*x**3 + 38*x**2 - 18*x + 3, 0)*

factor 1: 0.707106781186548
[0.707106781186548, 0.707106781186548]

coefs de m: 
-----------------------------------
factor 2: 0.48296291314453427
[0.482962913144534, 0.836516303737808, 0.224143868042013, -0.129409522551260]
-----------------------------------
factor 3: 0.332670552950083
[0.332670552950083, 0.806891509311093, 0.459877502118492, -0.135011020010255, -0.0854412738820267, 0.0352262918857095]
-----------------------------------
factor 4: 0.23037781330889
[0.230377813308897, 0.714846570552916, 0.630880767929859, -0.0279837694168599, -0.187034811719093, 0.0308413818355608, 0.0328830116668852, -0.0105974017850690]